# Lecture 17: NLP in Industry Domains — Answer Key
### NLP Course 2027 — Week 09

---
Complete answers for all four exercises in Lecture 17.

In [ ]:
import re
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
print('Ready')

---
## Exercise 17.1 — Tweet Sentiment Classifier

**Task:** Preprocess tweets and build a sentiment classifier. Document unique challenges.

**Key concept:** Tweets have unique noise: @mentions, #hashtags, URLs, emoji, abbreviations ('gr8', 'tbh'), and sarcasm. A tweet-specific preprocessor must handle each. `TweetTokenizer` from NLTK is aware of these patterns.

In [ ]:
# Simulated tweet dataset (expand to 100+ for real use)
tweets = [
    ('@Apple new iPhone is absolutely amazing!!! #Apple #iPhone', 'positive'),
    ('omg my iphone just died after 2 hours ugh #fail', 'negative'),
    ('lol cant believe how fast this phone charges gr8 job @Apple', 'positive'),
    ('iphone 16 camera is just ok tbh nothing special', 'neutral'),
    ('wtf apple why r ur prices so expensive!! boycott #ripoff', 'negative'),
    ('just got my new iphone and wow the display is gorgeous 😍', 'positive'),
    ('apple support is useless, waited 3 hours and got nothing', 'negative'),
    ('the new ios update is smooth and fast love it', 'positive'),
    ('meh. iphone same as last year tbh', 'neutral'),
    ('so frustrated, apple store is always crowded ugh', 'negative'),
    ('battery life on this iphone is incredible finally!! 🎉', 'positive'),
    ('apple is overrated. android does the same for half the price', 'negative'),
    ('idk maybe the new iphone is worth it idk', 'neutral'),
    ('best phone ive ever owned no question 💯', 'positive'),
    ('my iphone keeps crashing smh fix ur bugs @Apple', 'negative'),
] * 4  # ~60 examples

def preprocess_tweet(text):
    """Tweet-specific preprocessing pipeline."""
    text = re.sub(r'http\S+|www\S+', ' URL ', text)      # remove URLs
    text = re.sub(r'@\w+', ' USER ', text)                 # anonymize @mentions
    text = re.sub(r'#(\w+)', r' \1 ', text)               # expand hashtags
    text = re.sub(r'[^\x00-\x7F]+', ' EMOJI ', text)      # replace emoji
    # Expand common abbreviations
    abbrevs = {'gr8': 'great', 'tbh': 'to be honest', 'omg': 'oh my god',
               'wtf': 'what the', 'lol': 'laugh', 'idk': 'i do not know',
               'smh': 'shaking my head', 'ur': 'your', 'r': 'are'}
    for abbr, full in abbrevs.items():
        text = re.sub(rf'\b{abbr}\b', full, text, flags=re.IGNORECASE)
    return text.lower().strip()

texts  = [preprocess_tweet(t) for t, _ in tweets]
labels = [l for _, l in tweets]

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

pipe = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
                 ('clf',  LogisticRegression(max_iter=500, C=1.0))])
pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, preds):.4f}')
print(classification_report(y_test, preds))

print('Unique tweet challenges vs standard text:')
challenges = [
    '@mentions — noise or signal? (user reputation can predict sentiment)',
    '#hashtags — often carry sentiment directly (#fail, #love)',
    'Abbreviations — gr8, tbh, smh require a domain lexicon',
    'Emoji — strong sentiment signals; need special handling',
    'Sarcasm — "oh great, another bug" is negative, not positive',
    'Code-switching — mixing languages in one tweet',
]
for c in challenges:
    print(f'  • {c}')

---
## Exercise 17.2 — Dependency-Based ABSA with spaCy

**Task:** Use spaCy dependency parse to link adjectives to noun heads for ABSA.

**Key concept:** In a dependency parse, `amod` (adjectival modifier) links an adjective directly to the noun it modifies ('excellent camera' → camera←amod—excellent). This gives us aspect-opinion pairs without needing labeled training data.

In [ ]:
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')

    POS_ADJ = {'great', 'excellent', 'amazing', 'good', 'fantastic', 'fast',
               'beautiful', 'perfect', 'wonderful', 'brilliant', 'superb',
               'clear', 'bright', 'smooth', 'comfortable', 'reliable'}
    NEG_ADJ = {'terrible', 'awful', 'poor', 'slow', 'bad', 'horrible',
               'disappointing', 'broken', 'ugly', 'useless', 'worst',
               'expensive', 'heavy', 'annoying', 'cluttered'}

    def dependency_absa(review):
        doc     = nlp(review)
        results = []
        for token in doc:
            word = token.text.lower()
            # amod: adjective directly modifying a noun ("excellent camera")
            if token.dep_ == 'amod' and token.head.pos_ in ('NOUN', 'PROPN'):
                aspect    = token.head.text.lower()
                sentiment = ('positive' if word in POS_ADJ
                             else 'negative' if word in NEG_ADJ
                             else 'neutral')
                results.append({'aspect': aspect, 'opinion': word, 'sentiment': sentiment})
            # acomp: adjective as predicate complement ("the camera is excellent")
            elif token.dep_ == 'acomp':
                head = token.head  # verb like "is"
                # look for the subject noun
                for child in head.children:
                    if child.dep_ in ('nsubj', 'nsubjpass'):
                        aspect    = child.text.lower()
                        sentiment = ('positive' if word in POS_ADJ
                                     else 'negative' if word in NEG_ADJ
                                     else 'neutral')
                        results.append({'aspect': aspect, 'opinion': word, 'sentiment': sentiment})
        return results

    reviews = [
        'The camera quality is excellent but the battery life is terrible.',
        'Fast performance and great display, but the price is awful.',
        'The design is beautiful and the screen is bright.',
    ]
    for r in reviews:
        aspects = dependency_absa(r)
        print(f'Review: {r}')
        for a in aspects:
            print(f"  aspect={a['aspect']:<12} opinion={a['opinion']:<12} sentiment={a['sentiment']}")
        print()

except (ImportError, OSError) as e:
    print(f'spaCy unavailable: {e}')
    print('Install: pip install spacy && python -m spacy download en_core_web_sm')
    print()
    print('Expected output for "The camera quality is excellent but battery life is terrible.":')
    print('  aspect=quality      opinion=excellent   sentiment=positive')
    print('  aspect=life         opinion=terrible    sentiment=negative')

---
## Exercise 17.3 — BioBERT vs General BERT for Clinical NER

**Task:** Compare BioBERT and general BERT NER on a clinical note.

**Key concept:** Domain-adaptive pretraining (BioBERT trained on PubMed + PMC) gives a model vocabulary and contextual representations tuned to biomedical text. Medical terms like 'Metformin', 'myocardial infarction', 'Troponin' appear in the pretraining data and are tokenized more meaningfully.

In [ ]:
clinical_note = """
Patient: 58-year-old male presenting with chest pain and shortness of breath.
History: Hypertension (10 years), Type 2 Diabetes Mellitus.
Medications: Metformin 500mg twice daily, Lisinopril 10mg once daily.
Assessment: Rule out acute myocardial infarction (AMI).
Plan: ECG, Troponin levels, Aspirin 325mg stat, Cardiology consult.
"""

try:
    from transformers import pipeline

    print('General BERT NER (conll03):')
    general_ner = pipeline('ner', model='dbmdz/bert-large-cased-finetuned-conll03-english',
                           aggregation_strategy='simple')
    general_entities = general_ner(clinical_note)
    for e in general_entities:
        print(f"  [{e['entity_group']}] {e['word']}  (score={e['score']:.3f})")

    print()
    print('Note: General BERT misses medications (Metformin, Lisinopril, Aspirin)')
    print('and conditions (Hypertension, Diabetes, AMI) — these are not CoNLL entity types.')
    print()
    print('BioBERT NER (trained on biomedical text):')
    try:
        bio_ner = pipeline('ner', model='allenai/scibert_scivocab_cased',
                           aggregation_strategy='simple')
        bio_entities = bio_ner(clinical_note)
        for e in bio_entities:
            print(f"  [{e['entity_group']}] {e['word']}  (score={e['score']:.3f})")
    except Exception:
        print('  (BioBERT model not cached; expected output below)')
        print('  [DRUG] Metformin, Lisinopril, Aspirin')
        print('  [DISEASE] Hypertension, Type 2 Diabetes Mellitus, myocardial infarction')
        print('  [TEST] ECG, Troponin')

except Exception as e:
    print(f'Transformers not available: {e}')

print()
print('Key insight: Domain-specific pretraining (BioBERT) learns medical vocabulary')
print('that general models never see. This closes ~10–15 F1 points on biomedical NER.')

---
## Exercise 17.4 — Contract Clause Search System

**Task:** Given a query clause, find the top-3 most similar clauses using TF-IDF.

**Key concept:** Legal document retrieval uses the same TF-IDF cosine similarity as general IR, but domain vocabulary matters (indemnify, terminate, arbitration). Legal n-grams like 'binding arbitration' and 'governing law' are highly discriminative.

In [ ]:
clause_database = [
    ('This agreement shall be governed by the laws of the State of California.', 'governing_law'),
    ('The parties consent to exclusive jurisdiction in the courts of New York.', 'governing_law'),
    ('Either party may terminate with 30 days written notice.', 'termination'),
    ('This agreement terminates immediately upon material breach.', 'termination'),
    ('Each party shall indemnify the other from third-party claims.', 'indemnification'),
    ('Company shall defend and hold harmless the other party.', 'indemnification'),
    ('Payment is due net-30 from date of invoice.', 'payment'),
    ('Annual license fee of $5,000 payable in advance.', 'payment'),
    ('All information disclosed is confidential for 5 years.', 'confidentiality'),
    ('Receiving party shall not disclose any proprietary information.', 'confidentiality'),
    ('Disputes shall be resolved by binding arbitration in San Francisco.', 'dispute_resolution'),
    ('Any claim shall first be mediated before litigation.', 'dispute_resolution'),
]

clauses, labels = zip(*clause_database)

vec  = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
db_vecs = vec.fit_transform(clauses)

def search_clauses(query, top_k=3):
    qvec   = vec.transform([query])
    scores = cosine_similarity(qvec, db_vecs).flatten()
    top    = np.argsort(scores)[::-1][:top_k]
    return [(clauses[i], labels[i], scores[i]) for i in top]

queries = [
    'This contract is subject to the laws of Texas.',
    'The agreement may be ended with one month notice.',
    'Confidential data must not be shared with third parties.',
    'All disputes will go to arbitration.',
    'Fees are due within 30 days.',
]

for query in queries:
    print(f'Query: "{query}"')
    for clause, label, score in search_clauses(query):
        print(f'  [{label:<22}] (sim={score:.3f}) {clause}')
    print()

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 17.1 | Tweets need: URL removal, @anonymization, hashtag expansion, abbreviation normalization |
| 17.2 | Dependency ABSA: amod links adjective to noun aspect without labeled data |
| 17.3 | BioBERT: domain pretraining closes ~10–15 F1 on biomedical NER |
| 17.4 | Legal retrieval uses same TF-IDF mechanism; legal n-grams are highly discriminative |

---
*NLP Course 2027 — Week 09*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**